# LLM LoRA 학습 (Colab GPU)

- 목적: Mac(16GB)에서 학습할 수 없는 더 큰 베이스(예: `Qwen/Qwen3.5-2B`, `meta-llama/Llama-3.2-1B-Instruct`)를 LoRA 학습해 병합된 fp16 모델을 받는다. (PLAN LLM-07)
- 하는 일은 저장소의 `training/llm/train_lora.py`와 같다. 입력 형식은 `docs/decisions/2026-10-06-llm-input-format.md`.
- 런타임: **GPU** (A100·L4면 bf16, T4면 fp32로 학습한다)
- **2026-10-06 기준 Colab에서 실행해 보지 않았다.** 막힌 곳을 이 노트북과 이슈에 남긴다.
- 결과 모델은 git에 올리지 않는다. zip으로 받아 `docs/artifacts.md` 등록부에 한 줄 남긴다.

In [ ]:
!nvidia-smi -L
!git clone --depth 1 https://github.com/samsung-team-jarvis/jarvis.git
%cd jarvis
!git log --oneline -1  # 이 커밋을 학습 기록에 남긴다

In [ ]:
# 저장소와 같은 버전으로 맞춘다 (torch를 다시 설치해 몇 분 걸린다)
!pip install -q -r training/llm/requirements-eval.txt

Llama처럼 승인이 필요한 모델이면 Hugging Face 토큰으로 로그인한다 (토큰을 셀에 적어 두지 않는다).

In [ ]:
# from huggingface_hub import login
# login()

In [ ]:
MODEL = "Qwen/Qwen3.5-2B"  # 베이스 모델 (Instruct)
NAME = "llm_qwen3.5-2b_jarvis_v1"  # docs/artifacts.md 이름 규칙
!python -m training.llm.train_lora --model {MODEL} --name {NAME} --limit 32 --epochs 1  # 먼저 끝까지 도는지

In [ ]:
!python -m training.llm.train_lora --model {MODEL} --name {NAME}
!cat runs/llm/{NAME}/train_log.json

In [ ]:
# 학습한 모델로 val 평가 (test는 비교할 때 한 번만)
!python -m bench.llm_eval --engine hf --finetuned --model runs/llm/{NAME} --split val --csv runs/llm/{NAME}_val.csv

In [ ]:
!cd runs/llm && zip -qr {NAME}.zip {NAME} {NAME}_val.csv && ls -lh {NAME}.zip
from google.colab import files

files.download(f"runs/llm/{NAME}.zip")